# 1 Painel de controle dos fluxos
Mostra a última execução de cada fluxo. O NULLIF evita divisão por zero quando um fluxo não processou nada.

In [0]:
%sql
SELECT fluxo, tabela_silver, registros_lidos, registros_validos, registros_quarentena,
       ROUND(100 * registros_quarentena / NULLIF(registros_validos + registros_quarentena, 0), 2) AS pct_quarentena,
       ultimo_ingerido_em, atualizado_em
FROM corretora.ops.controle_silver
ORDER BY pct_quarentena DESC, fluxo;

# 2 Unicidade das chaves
A garantia mais importante da Silver: uma linha por chave de negócio.
Qualquer DUPLICADA é bug: na deduplicação, na escolha da chave ou no MERGE. Nunca siga com chave duplicada na Silver, porque na Gold ela vira valor somado em dobro.

In [0]:
CHAVES = {
    'cotacoes_intradiarias': ['ativo', 'fonte', 'cotado_em_utc'],
    'precos_diarios': ['ativo', 'fonte', 'data_pregao'],
    'ativos': ['ativo'],
    'indicadores_economicos': ['codigo_sgs', 'data_referencia'],
    'tesouro_precos_taxas': ['tipo_titulo', 'data_vencimento', 'data_base'],
    'clientes': ['id_cliente'],
    'operacoes': ['id_operacao'],
}

resultado = []
for tabela, chaves in CHAVES.items():
    df = spark.table(f'corretora.silver.{tabela}')
    total = df.count()
    duplicadas = total - df.select(*chaves).distinct().count()
    resultado.append((tabela, total, duplicadas, 'OK' if duplicadas == 0 else 'DUPLICADA'))

display(spark.createDataFrame(
    resultado, 'tabela string, registros long, chaves_duplicadas long, status string'))

# 3 O que foi para a quarentena
Para cada regra com ocorrências, a pergunta é: o dado está errado, ou a regra está errada? As duas respostas são possíveis, e a decisão é sua como dono do dado.

In [0]:
%sql
SELECT tabela_destino, regra, COUNT(*) AS registros, MAX(quarentenado_em) AS ultima_ocorrencia
FROM corretora.silver.quarentena
LATERAL VIEW explode(regras_violadas) t AS regra
GROUP BY ALL
ORDER BY registros DESC;

Para ver os registros de uma regra específica:

In [0]:
%sql
SELECT regras_violadas, registro
FROM corretora.silver.quarentena
WHERE array_contains(regras_violadas, 'faixa_invalida')
LIMIT 20;

# 4 Reconciliação Bronze x Silver
O esperado é sem_destino = 0. Um valor positivo significa que a Silver ainda não processou as operações mais recentes da Bronze; rode o 21_silver e confira de novo.

Nas operações, toda operação da Bronze precisa estar ou na Silver ou na quarentena:

In [0]:
%sql
WITH bronze AS (
  SELECT COUNT(DISTINCT id_operacao) AS n FROM corretora.bronze.operacoes
),
silver AS (
  SELECT COUNT(*) AS n FROM corretora.silver.operacoes
),
quarentena AS (
  SELECT COUNT(DISTINCT get_json_object(registro, '$.id_operacao')) AS n
  FROM corretora.silver.quarentena
  WHERE fluxo = 'operacoes'
)
SELECT b.n AS bronze, s.n AS silver, q.n AS quarentena, b.n - s.n - q.n AS sem_destino
FROM bronze b CROSS JOIN silver s CROSS JOIN quarentena q;

# 5 Change Data Feed

Cada atualização de cliente aparece como um par update_preimage (antes) e update_postimage (depois). No Nível 3, a Gold vai ler só essas mudanças em vez da tabela inteira.

In [0]:
%sql
SELECT _change_type, COUNT(*) AS eventos
FROM table_changes('corretora.silver.clientes', 1)
GROUP BY _change_type;

# 6 Sanidade de negócio
Última cotação de cada ativo, já com a classe vinda do catálogo:

In [0]:
%sql
SELECT a.classe, c.ativo, c.fonte, c.preco_compra, c.preco_venda, c.cotado_em_utc
FROM corretora.silver.cotacoes_intradiarias c
LEFT JOIN corretora.silver.ativos a ON a.ativo = c.ativo
QUALIFY ROW_NUMBER() OVER (PARTITION BY c.ativo ORDER BY c.cotado_em_utc DESC) = 1
ORDER BY a.classe, c.ativo;

Último valor de cada indicador, agora com nome e unidade:

In [0]:
%sql
SELECT nome_serie, unidade, data_referencia, valor
FROM corretora.silver.indicadores_economicos
QUALIFY ROW_NUMBER() OVER (PARTITION BY codigo_sgs ORDER BY data_referencia DESC) = 1
ORDER BY nome_serie;